In [9]:
# =======================================================================
# CELL 1: Data Ingestion & Native Edge Math Engine (Dart Replica)
# =======================================================================
import numpy as np
import pandas as pd

# RELATIVE PATH: Reaches out of 'validation_scripts' into 'data'
csv_path = "../data/morph_analysis_data/theia_poblacion_03-11-2025_17-15_157-especimenes.csv"

# --- 1. Load Dataset ---
df = pd.read_csv(csv_path)
if 'image_name' not in df.columns:
    raise ValueError("CSV must contain an 'image_name' column.")

specimen_names = df['image_name'].astype(str).values
coord_df = df.drop(columns='image_name')

coords_matrix = coord_df.to_numpy(dtype=np.float64, copy=True)
n_specimens = coords_matrix.shape[0]
n_landmarks = coords_matrix.shape[1] // 2

data_array = coords_matrix.reshape((n_specimens, n_landmarks, 2))
data_array_r = np.transpose(data_array, (1, 2, 0)) # For R

print("="*70)
print(" DATA INGESTION COMPLETED")
print("="*70)
print(f" Total Specimens : {n_specimens}")
print(f" Landmarks       : {n_landmarks}")

# --- 2. Native Edge Procrustes (GPA) Engine (Replicating Dart/ml_linalg) ---
def _center(shape):
    return shape - shape.mean(axis=0)

def _centroid_size(shape):
    return np.sqrt((shape**2).sum())

def _normalize_frobenius(shape):
    n = np.sqrt((shape**2).sum())
    return shape if n <= 1e-12 else (shape / n)

def _rotation_matrix(X, Y):
    C = X.T @ Y
    U, _, VT = np.linalg.svd(C)
    R = U @ VT
    if np.linalg.det(R) < 0:
        VT_corr = VT.copy()
        VT_corr[-1, :] *= -1.0
        R = U @ VT_corr
    return R

def run_gpa_edge(data_array, max_iter=10):
    n, p, k = data_array.shape
    scaled = np.zeros_like(data_array)
    c_sizes = np.zeros(n)
    
    for i in range(n):
        centered = _center(data_array[i])
        cs = _centroid_size(centered)
        c_sizes[i] = cs
        scaled[i] = centered if cs <= 1e-12 else (centered / cs)

    aligned = scaled.copy()
    reference = _normalize_frobenius(np.mean(aligned, axis=0))

    for _ in range(max_iter):
        next_aligned = np.zeros_like(aligned)
        for i in range(n):
            R = _rotation_matrix(scaled[i], reference)
            next_aligned[i] = scaled[i] @ R
        aligned = next_aligned
        reference = _normalize_frobenius(np.mean(aligned, axis=0))
        
    aligned[..., [0, 1]] = aligned[..., [1, 0]]
    mean_shape = np.mean(aligned, axis=0)
    return aligned, mean_shape, c_sizes

# --- 3. Native Edge PCA Engine (Replicating Dart/ml_linalg) ---
def run_pca_edge(aligned_shapes, k_components=3):
    n, p, _ = aligned_shapes.shape
    d = 2 * p
    X = np.zeros((n, d), dtype=float)
    for i in range(n):
        X[i] = aligned_shapes[i].reshape(-1, order='C')

    mean_vec = X.mean(axis=0)
    Xc = X - mean_vec
    cov = (Xc.T @ Xc) / max(1, n - 1)

    evals, evecs = np.linalg.eigh(cov)
    order = np.argsort(evals)[::-1]
    evals = evals[order]
    evecs = evecs[:, order]

    kk = min(k_components, d)
    scores = Xc @ evecs[:, :kk]
    explained_pct = (evals[:kk] / (evals.sum() or 1.0)) * 100.0
    return scores, explained_pct

# Run analytical engine
gpa_edge, mean_edge, csize_edge = run_gpa_edge(data_array)
pca_scores_edge, pca_var_edge = run_pca_edge(gpa_edge)
print("[OK] Native Edge Analytical Engine Executed.")

 DATA INGESTION COMPLETED
 Total Specimens : 157
 Landmarks       : 32
[OK] Native Edge Analytical Engine Executed.


In [10]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [11]:
%%R -i data_array_r -o gpa_coords_r -o consensus_shape_r -o pca_scores_r -o pc_var_r -o csize_r
# =======================================================================
# CELL 2: Geomorph Analysis (R Gold Standard)
# =======================================================================
options(warn=-1)

if (!requireNamespace("geomorph", quietly = TRUE)) {
  install.packages("geomorph", repos = "https://cloud.r-project.org")
}
suppressPackageStartupMessages(library(geomorph))

# 1) GPA
gpa_results <- gpagen(data_array_r, print.progress = FALSE)
csize_r <- gpa_results$Csize

# 2) PCA
pca_results <- gm.prcomp(gpa_results$coords)

# 3) Export to Python
gpa_coords_r      <- gpa_results$coords               
consensus_shape_r <- gpa_results$consensus            
pca_scores_r      <- pca_results$x[, 1:3, drop = FALSE]  

pc_var_r <- (pca_results$sdev^2) / sum(pca_results$sdev^2) * 100
if (length(pc_var_r) >= 3) {
  pc_var_r <- pc_var_r[1:3]
} else {
  pc_var_r <- c(pc_var_r, rep(NA_real_, 3 - length(pc_var_r)))
}

In [12]:
# =======================================================================
# CELL 3: Algorithmic Fidelity Report (MEE Manuscript Format)
# =======================================================================
import pandas as pd
from scipy.stats import pearsonr

r_scores = np.array(pca_scores_r)
py_scores = np.array(pca_scores_edge)

# --- 1. Eigenvector Alignment (Sign Invariance Correction) ---
py_aligned = py_scores.copy()
for j in range(min(3, r_scores.shape[1], py_aligned.shape[1])):
    c1 = np.corrcoef(r_scores[:, j], py_aligned[:, j])[0, 1]
    c2 = np.corrcoef(r_scores[:, j], -py_aligned[:, j])[0, 1]
    if abs(c2) > abs(c1):
        py_aligned[:, j] = -py_aligned[:, j]

r_csize_val, _ = pearsonr(np.array(csize_r).flatten(), csize_edge)

# --- 2. FORMATTED MANUSCRIPT OUTPUT ---
print("\n" + "="*85)
print(" ALGORITHMIC FIDELITY: Theia Native Engine vs. geomorph (R)")
print(" (Empirical metrics extraction for Methods in Ecology and Evolution)")
print("="*85)

print("\n[A] EXPLAINED VARIANCE RATIO")
print("-" * 85)
print(f"  PC1 Variance (Theia Engine) : {pca_var_edge[0]:.2f}%")
print(f"  PC1 Variance (geomorph R)   : {pc_var_r[0]:.2f}%")
print(f"  PC2 Variance (Theia Engine) : {pca_var_edge[1]:.2f}%")
print(f"  PC2 Variance (geomorph R)   : {pc_var_r[1]:.2f}%")

print("\n[B] MATHEMATICAL CORRELATION (Pearson's r)")
print("-" * 85)
print(f"  Centroid Size (Csize)       : |r| = {abs(r_csize_val):.8f}")
for j in range(min(3, r_scores.shape[1], py_scores.shape[1])):
    r_val = np.corrcoef(r_scores[:, j], py_scores[:, j])[0, 1]
    sign = "+" if r_val >= 0 else "-"
    print(f"  Principal Component {j+1} (PC{j+1}) : |r| = {np.abs(r_val):.8f} (Original Sign: {sign})")

print("\n[C] SPATIAL TOPOLOGY SAMPLE (First 5 Specimens)")
print("-" * 85)
n_show = min(5, r_scores.shape[0])
df_pca = pd.DataFrame({
    'Specimen': specimen_names[:n_show],
    'R_PC1': np.round(r_scores[:n_show, 0], 6), 'Theia_PC1': np.round(py_aligned[:n_show, 0], 6),
    'R_PC2': np.round(r_scores[:n_show, 1], 6), 'Theia_PC2': np.round(py_aligned[:n_show, 1], 6),
})
print(df_pca.to_string(index=False))

print("\n  EDITORIAL INTERPRETATION:")
print("  The near-perfect variance equivalence (~51%) and absolute principal component")
print("  correlation (|r| > 0.999) demonstrate that Theia's sovereign mathematical engine")
print("  (replicated here natively in Python) perfectly mirrors the geomorph (R) standard.")
print("="*85 + "\n")


 ALGORITHMIC FIDELITY: Theia Native Engine vs. geomorph (R)
 (Empirical metrics extraction for Methods in Ecology and Evolution)

[A] EXPLAINED VARIANCE RATIO
-------------------------------------------------------------------------------------
  PC1 Variance (Theia Engine) : 50.91%
  PC1 Variance (geomorph R)   : 51.02%
  PC2 Variance (Theia Engine) : 23.73%
  PC2 Variance (geomorph R)   : 23.78%

[B] MATHEMATICAL CORRELATION (Pearson's r)
-------------------------------------------------------------------------------------
  Centroid Size (Csize)       : |r| = 1.00000000
  Principal Component 1 (PC1) : |r| = 0.99999974 (Original Sign: -)
  Principal Component 2 (PC2) : |r| = 0.99999896 (Original Sign: -)
  Principal Component 3 (PC3) : |r| = 0.99999746 (Original Sign: -)

[C] SPATIAL TOPOLOGY SAMPLE (First 5 Specimens)
-------------------------------------------------------------------------------------
    Specimen     R_PC1  Theia_PC1     R_PC2  Theia_PC2
DSCN6613.jpg -0.058433   